# 🟡 L04 — Optional Extensions
**⏱️ Optional — no suggested time. Dip into any section that interests you.**

> *Pure self-study material. Skipping this will not affect any later lesson.*

Three independent sections:

1. **Ridge / Lasso / ElasticNet** — regularisation for linear models
2. **Gini vs Entropy** — the two split criteria for decision trees, and why they almost never matter
3. **GridSearch vs RandomizedSearch** — head-to-head comparison on the same model

---

## Setup

In [ ]:
# --- Setup: toolbox + the familiar NorthStar data and prep recipe ---
import numpy as np                     # calculator for lists of numbers
import pandas as pd                    # tables of data
import matplotlib.pyplot as plt        # charts
import warnings                        # hide distracting warnings
import time                            # stopwatch for comparing search speeds

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold, GridSearchCV, RandomizedSearchCV  # splitting + fair scoring + two kinds of auto-tuning
from sklearn.compose import ColumnTransformer      # different prep per column type
from sklearn.pipeline import Pipeline              # prep + model assembly line
from sklearn.preprocessing import StandardScaler, OneHotEncoder  # scale numbers; encode categories
from sklearn.impute import SimpleImputer           # fill missing values
from sklearn.linear_model import LogisticRegression, RidgeClassifier  # linear models for the regularisation demo
from sklearn.tree import DecisionTreeClassifier    # single tree (for the gini/entropy demo)
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier  # the ensembles
from sklearn.metrics import f1_score               # our scorecard

warnings.filterwarnings("ignore")   # keep output clean

# Re-load NorthStar churn + standard split (identical to the class notebooks)
df = pd.read_csv("data/northstar_churn.csv")
y  = df["churned"]                                # the answer
X  = df.drop(columns=["customer_id", "churned"])  # the clues
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=42,
)

# Number vs category columns
numeric_features = ["age", "tenure_months", "num_purchases_quarter",
                    "avg_monthly_spend_gbp", "returns_per_purchase",
                    "last_login_days_ago", "avg_review_polarity",
                    "support_tickets_quarter"]
categorical_features = ["region", "subscription_tier"]

# The usual prep recipe
preprocessor = ColumnTransformer([
    ("num", Pipeline([("imp", SimpleImputer(strategy="median")),
                      ("scl", StandardScaler())]), numeric_features),
    ("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")),
                      ("enc", OneHotEncoder(handle_unknown="ignore", sparse_output=False))]),
                      categorical_features),
])

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)  # fair 5-fold test
print("✅ Setup complete.")

# 1. Ridge / Lasso / ElasticNet

`LogisticRegression` uses L2 regularisation by default — that's *Ridge* in disguise. Switching to L1 gives you *Lasso*. Using both gives you *ElasticNet*.

| Penalty | What it does | When |
|---|---|---|
| **L2 (Ridge)** | Shrinks all coefficients toward zero. Never to exactly zero. | Default. Most algorithms. |
| **L1 (Lasso)** | Shrinks some coefficients to EXACTLY zero. Feature selection. | When you have many features and want to identify the important ones. |
| **L1 + L2 (ElasticNet)** | Mix of both. | When features are correlated — Lasso alone can pick arbitrary ones. |

The `C` parameter is the *inverse* of regularisation strength. Small C = strong penalty = simpler model. Large C = weak penalty = closer to unregularised.

In [ ]:
# --- Compare four flavours of regularisation on logistic regression ---
# Regularisation = deliberately shrinking the model's coefficients so it can't over-trust any one feature.
configs = [
    ("No regularisation",  LogisticRegression(penalty=None, max_iter=2000, class_weight="balanced", random_state=42)),
    ("Ridge (L2)",          LogisticRegression(penalty="l2", C=1.0, max_iter=2000, class_weight="balanced", random_state=42)),          # shrink all coefficients
    ("Lasso (L1)",          LogisticRegression(penalty="l1", C=1.0, solver="liblinear", max_iter=2000, class_weight="balanced", random_state=42)),  # shrink AND zero-out some
    ("ElasticNet (50/50)",  LogisticRegression(penalty="elasticnet", C=1.0, l1_ratio=0.5,
                                                solver="saga", max_iter=2000, class_weight="balanced", random_state=42)),               # half-and-half mix
]

records = []
for name, model in configs:
    pipe = Pipeline([("prep", preprocessor), ("model", model)])   # prep + this model
    cv_f1 = cross_val_score(pipe, X_train, y_train, cv=cv, scoring="f1", n_jobs=-1).mean()  # fair score
    # Number of features whose coefficient is exactly zero (= features the model IGNORES entirely)
    pipe.fit(X_train, y_train)
    coefs = pipe.named_steps["model"].coef_[0]
    n_zero = int((np.abs(coefs) < 1e-6).sum())
    records.append((name, cv_f1, n_zero, len(coefs)))

# Show all four side by side
print(pd.DataFrame(records, columns=["penalty", "cv_f1", "n_zero_coefs", "n_total_coefs"]).to_string(index=False))
print()
print("→ Lasso typically drops some features to zero (built-in feature selection).")
print("→ Ridge keeps all features but shrinks the small ones.")
print("→ ElasticNet does some of both.")

## Tuning C with GridSearchCV

The right `C` depends on the dataset. Search a log-spaced grid.

In [ ]:
# --- How strong should the regularisation be? Sweep C from very strong to very weak ---
# Note: C is INVERSE strength — small C = strong shrinking, big C = barely any
C_grid = [0.001, 0.01, 0.1, 1.0, 10.0, 100.0]
records_C = []

for C_val in C_grid:
    # Same model, only C changes
    pipe = Pipeline([
        ("prep",  preprocessor),
        ("model", LogisticRegression(penalty="l2", C=C_val, max_iter=2000,
                                       class_weight="balanced", random_state=42)),
    ])
    f1 = cross_val_score(pipe, X_train, y_train, cv=cv, scoring="f1", n_jobs=-1).mean()  # fair score
    records_C.append((C_val, f1))

C_df = pd.DataFrame(records_C, columns=["C", "cv_f1"])   # results table
print(C_df.to_string(index=False))

# Plot the sweep on a log axis (C spans 5 orders of magnitude)
fig, ax = plt.subplots(figsize=(10, 4.5))
ax.semilogx(C_df["C"], C_df["cv_f1"], "o-", linewidth=2, color="steelblue", markersize=10)
ax.set_xlabel("C (inverse regularisation strength)")
ax.set_ylabel("CV F1")
ax.set_title("Logistic Regression — CV F1 vs C")
plt.tight_layout()
plt.show()

# Report the best C found
best = C_df.iloc[C_df["cv_f1"].idxmax()]
print(f"\nBest C: {best['C']}  → CV F1: {best['cv_f1']:.3f}")

**Takeaway:** for this dataset and class-balanced logistic regression, C in the 0.1–10 range is fine. Tuning rarely moves the F1 by more than 0.01.

# 2. Gini vs Entropy — the split criteria

A decision tree picks each split by maximising IMPURITY REDUCTION. The two standard impurity measures:

- **Gini:** `1 - Σ p_i^2`  (faster to compute, default in sklearn)
- **Entropy:** `-Σ p_i · log2(p_i)`  (slightly more theoretically grounded)

They give VERY similar results in practice. We're going to verify that.

In [ ]:
# --- Gini vs Entropy: does the tree's split-quality formula matter? ---
# Same forest twice — the ONLY difference is the impurity formula used to pick splits
rf_gini = Pipeline([("prep", preprocessor),
                    ("model", RandomForestClassifier(n_estimators=200, min_samples_leaf=5,
                                                     criterion="gini",        # the default formula
                                                     class_weight="balanced",
                                                     n_jobs=-1, random_state=42))])
rf_entropy = Pipeline([("prep", preprocessor),
                       ("model", RandomForestClassifier(n_estimators=200, min_samples_leaf=5,
                                                         criterion="entropy", # the alternative formula
                                                         class_weight="balanced",
                                                         n_jobs=-1, random_state=42))])

# Score both with the same fair 5-fold test
f1_gini    = cross_val_score(rf_gini,    X_train, y_train, cv=cv, scoring="f1", n_jobs=-1).mean()
f1_entropy = cross_val_score(rf_entropy, X_train, y_train, cv=cv, scoring="f1", n_jobs=-1).mean()

print(f"criterion='gini':    CV F1 = {f1_gini:.4f}")
print(f"criterion='entropy': CV F1 = {f1_entropy:.4f}")
print(f"Difference: {abs(f1_gini - f1_entropy):.4f}")
print()
print("→ Difference is in the third decimal place. Not worth tuning.")
print("→ Stick with the default (gini) unless you have a specific reason not to.")

# 3. GridSearchCV vs RandomizedSearchCV

`GridSearchCV` tries every combination in your grid. `RandomizedSearchCV` samples N combinations from a distribution. For large grids, randomized search finds combinations close to the grid optimum at a fraction of the compute.

Let's compare both on the same gradient boosting pipeline with a fairly large parameter space.

In [ ]:
# --- Grid search (try everything) vs randomized search (sample a few) at similar budgets ---
gb_pipe = Pipeline([
    ("prep",  preprocessor),
    ("model", HistGradientBoostingClassifier(class_weight="balanced", random_state=42)),
])

# A larger grid that we'd never want to enumerate fully
big_grid = {
    "model__learning_rate": [0.01, 0.03, 0.05, 0.08, 0.1, 0.15, 0.2, 0.3],
    "model__max_iter":      [100, 200, 400, 800],
    "model__max_depth":     [3, 4, 5, 6, 8, None],
    "model__min_samples_leaf": [10, 20, 50],
}
n_combos = 8 * 4 * 6 * 3   # how many combinations the full menu holds
print(f"Full grid: {n_combos} combinations × 5-fold = {n_combos*5} model fits — would take a while.")
print()

# Approach A: GridSearchCV on a SUBSET of the grid (8 combos, tried exhaustively)
small_grid = {
    "model__learning_rate": [0.05, 0.1],
    "model__max_iter":      [200, 400],
    "model__max_depth":     [4, 6],
}
start = time.time()   # stopwatch A
grid_search = GridSearchCV(gb_pipe, small_grid, cv=cv, scoring="f1", n_jobs=-1)
grid_search.fit(X_train, y_train)
grid_time = time.time() - start

# Approach B: RandomizedSearchCV on the FULL grid, sampling 25 combos at random
start = time.time()   # stopwatch B
random_search = RandomizedSearchCV(
    gb_pipe, param_distributions=big_grid, n_iter=25,
    cv=cv, scoring="f1", n_jobs=-1, random_state=42,
)
random_search.fit(X_train, y_train)
random_time = time.time() - start

# Compare scores and wall-clock time
print(f"GridSearchCV   ({len(grid_search.cv_results_['params']):>2} combos, full enumeration of small grid):")
print(f"   Best CV F1: {grid_search.best_score_:.3f}")
print(f"   Time: {grid_time:.1f}s")
print(f"   Best: {grid_search.best_params_}")
print()
print(f"RandomizedSearchCV ({len(random_search.cv_results_['params']):>2} combos, random sample of big grid):")
print(f"   Best CV F1: {random_search.best_score_:.3f}")
print(f"   Time: {random_time:.1f}s")
print(f"   Best: {random_search.best_params_}")

### 💡 What this tells us

- **RandomizedSearchCV often matches or beats GridSearchCV** at the same compute budget — because it explores the parameter space more broadly.
- **GridSearchCV is fine when the grid is small** (under ~30 combinations) — it's exhaustive and deterministic.
- **For larger searches, prefer RandomizedSearch** — and consider Bayesian search (e.g., `optuna`) for even better performance.

**Rule of thumb:**
- ≤30 combinations → GridSearchCV
- 30–500 combinations → RandomizedSearchCV with `n_iter=30-50`
- > 500 combinations → Bayesian optimisation library (optuna, hyperopt)

---

## Closing thoughts

You've now seen:
- **Regularisation** — Ridge/Lasso/ElasticNet for linear models (rarely the bottleneck on tree-based pipelines)
- **Tree split criteria** — gini vs entropy give nearly identical results; don't tune
- **Search strategies** — Grid for small spaces, Random for large spaces

Two patterns from this notebook that you'll keep using:
1. **Log-scale sweeps for continuous hyperparameters** (`C`, `learning_rate`)
2. **Time-budget the search** — fit count × CV folds = total cost. Pick a search that fits.